# Distance from Shore, Seasonality, and Low-Oxygen Conditions in NYC Harbor

My hypothesis is that shore-clustered cyanobacteria, present in hot summer conditions with greater incident sunlight (due to silicate solubility proportionte to temperature driving dominance in summer months) contribute more to hypoxic conditions than diatoms, which are further from shore typically and predominate in winter months.

**Research question:** Do warm-season, nearshore plankton-rich conditions in NYC Harbor correspond with hypoxic events events than cooler, farther-from-shore conditions?

**NOTE:** There are a few important checks here -- bloom proxies will be treated as general phytoplankton, as no dataset can accurately describe order from chlorophyll-a proxy alone. 

Overall, I would predict that summer observations closer to shore will tend to have higher chlorophyll values, which would co-occur with lower bottom dissolved oxygen per site, while winter observations will show a different mean weighted distance from show and fewer hypoxic events as a result.

**Datasets Used**:
* NYC Harbor Water Quality Monitoring Data (https://data.cityofnewyork.us/Environment/Harbor-Water-Quality/5uug-f49n)
* Exact fields intended for use from the harbor dataset: `sampling_location`, `sample_date`, `top_sample_temperature_c`, `ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l`, `top_active_chlorophyll_a_g_l`, `top_silica_mg_l`, `long`, `lat`
* NY boundary geometry: `State_Shoreline.shp`
* NJ boundary geometry: `NJ_State_Boundary.shp`


#### Methods

This project uses NYC Harbor water-quality observations together with NY/NJ boundary shapefiles to estimate each sampling point’s distance from shore. Chlorophyll-a is treated as a general phytoplankton proxy, temperature is used as the main seasonal driver, and bottom dissolved oxygen is used to identify low-oxygen conditions. 

###### The Overall work flow is to:
1. Use Indexing / Slicing to select the relevant variables from the harbor dataset via `groupby`
2. Convert Sampling points into a spatial later (via xxx), by esimating distance from each point to nearest polygon edge (`geopandas`)
3. Classify observations by season and low-oxygen status (`numpy.where` to define categorical variables from continuous fields)
3. Use `matplotlib` to visualize shoreline position, monthly distance trends, and oxygen correlation


In [5]:
# Important packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path


In [6]:
# Set file paths
# NOTE: NOAA CUST Shorelines ended up being massive, so instead, point-on-edge distance detection is just going to be 
# the method I'm stickign with for TIGER shapefiles...
water_fp = Path('NYS-Harbor-Data.csv')
ny_shore_fp = Path('State_Shoreline.shp')
nj_shore_fp = Path('NJ_State_Boundary.shp')


In [7]:
# Loading the specific fields needed from the harbor dataset
use_cols = [
    'sampling_location',
    'sample_date',
    'top_sample_temperature_c',
    'ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l',
    'top_active_chlorophyll_a_g_l',
    'top_silica_mg_l',
    'long',
    'lat'
]

df = pd.read_csv(water_fp, usecols=use_cols)
df.head()


,sampling_location,sample_date,top_sample_temperature_c,ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l,top_silica_mg_l,top_active_chlorophyll_a_g_l,long,lat
0,BB2,2006-08-09T00:00:00.000,26.47,0.03,1.3,37.28,40.66017,-73.82217
1,BB2,2006-09-12T00:00:00.000,21.88,4.54,2.16,14.60,40.66033,-73.82183
2,BB4,2006-08-09T00:00:00.000,27.37,1.53,0.87,48.51,40.65183,-73.82317
3,BB4,2006-09-12T00:00:00.000,21.51,5.23,2.11,16.28,40.65133,-73.82300
4,E10,2006-10-23T00:00:00.000,16.01,6.68,0.99,2.57,40.84317,-73.76500


Now that the relevant harbor fields are loaded, I just simplified the names and converted the raw date and measurement fields into something easier to work with:

In [8]:
# Renaming long field names, convert date
df = df.rename(columns={
    'sampling_location': 'site',
    'sample_date': 'date',
    'top_sample_temperature_c': 'temp_c',
    'ctd_conductivity_temperature_depth_profiler_bottom_dissolved_oxygen_mg_l': 'bottom_do_mg_l',
    'top_active_chlorophyll_a_g_l': 'chl_a',
    'top_silica_mg_l': 'silica_mg_l',
    'long': 'lon',
    'lat': 'lat'
})
# Conv to date
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month

# Setting numeric
for c in ['temp_c', 'bottom_do_mg_l', 'chl_a', 'silica_mg_l', 'lon', 'lat']:
    df[c] = pd.to_numeric(df[c])

df = df.dropna(subset=['temp_c', 'bottom_do_mg_l', 'chl_a', 'lon', 'lat'])

df.head()

ValueError: Unable to parse string "QD" at position 460

In [ ]:
The harbor data are still just tabular observations at this point. To measure distance from shore, the sampling coordinates need to be turned into spatial points, and the NY/NJ polygon layers need to be loaded into the same CRS.

In [ ]:
# Load NY and NJ geometry files
ny_shore = gpd.read_file(ny_shore_fp)
nj_shore = gpd.read_file(nj_shore_fp)

# reference layer comb
shore = pd.concat([ny_shore, nj_shore], ignore_index=True)
shore = gpd.GeoDataFrame(shore, geometry='geometry', crs=ny_shore.crs)

# sampling coordinates --> GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326'
)

#reprojections (m instead of degrees because I am lazy)
gdf = gdf.to_crs('EPSG:32618')
shore = shore.to_crs('EPSG:32618')

Next I'm defining soem categorical variables to be compared later in summary
(season, low-oxygen status, and a simple nearshore/offshore split based on the median shoreline distance).

In [ ]:
gdf['season'] = np.where( #obs per season
    gdf['month'].isin([12, 1, 2]), 'winter', # diatoms (hopefully...)
    np.where(gdf['month'].isin([6, 7, 8]), 'summer', 'transition') #(cyano)
)

# I'm considering hypoxic conditiong to be anywhere below 3.0 mg/L
gdf['low_oxygen_3mg'] = np.where(gdf['bottom_do_mg_l'] < 3.0, 1, 0)

# nearshore vs offshore using median distance
dist_thresh = gdf['distance_to_shore_m'].median()
gdf['shore_bin'] = np.where(gdf['distance_to_shore_m'] < dist_thresh, 'nearshore', 'offshore')

gdf[['season', 'low_oxygen_3mg', 'shore_bin']].head()

Before comparing oxygen and chlorophyll directly, it makes sense to first check whether the average distance from shore itself changes through the year.

In [ ]:
# Placeholder summary table
monthly = gdf.groupby('month', as_index=False).agg(
    mean_distance_m=('distance_to_shore_m', 'mean'),
    mean_chl=('chl_a', 'mean'),
    mean_temp=('temp_c', 'mean'),
    mean_bottom_do=('bottom_do_mg_l', 'mean'),
    low_oxygen_freq=('low_oxygen_3mg', 'mean')
)

monthly


In [ ]:
# Placeholder plot
plt.figure(figsize=(7, 4))
plt.plot(monthly['month'], monthly['mean_distance_m'], marker='o')
plt.title('Mean Distance from Shore by Month')
plt.xlabel('Month')
plt.ylabel('Mean Distance from Shore (m)')
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.show()
# Mean sampling distance from show does appear to vary from month to month!

I'm mainly concerned about the comparison between summer and winter, so now I will be looking at these two seasonal regiemes as well (`season_space`) 

In [ ]:
# focus only on winter and summer
subset = gdf[gdf['season'].isin(['winter', 'summer'])]

season_space = subset.groupby(['season', 'shore_bin'], as_index=False).agg(
    mean_chl=('chl_a', 'mean'),
    mean_temp=('temp_c', 'mean'),
    mean_bottom_do=('bottom_do_mg_l', 'mean'),
    mean_silica=('silica_mg_l', 'mean'),
    low_oxygen_freq=('low_oxygen_3mg', 'mean'),
    mean_distance_m=('distance_to_shore_m', 'mean')
)

season_space

In [ ]:
# map!
fig, ax = plt.subplots(figsize=(7, 7))
shore.plot(ax=ax, color='none', edgecolor='black', linewidth=0.5)
gdf.plot(ax=ax, column='low_oxygen_3mg', markersize=7, legend=True)

ax.set_title('NYC Harbor Sampling Points and Low-Oxygen Observations')
ax.set_axis_off()
plt.show()

Now to compare the main variables of interest -- chlorophyll, temperature, and bottom dissolved oxygen -- across the nearshore/offshore and winter/summer categories:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

vars_to_plot = [
    ('mean_chl', 'Mean Chlorophyll-a'),
    ('mean_temp', 'Mean Temperature (C)'),
    ('mean_bottom_do', 'Mean Bottom DO (mg/L)')
]

for ax, (v, title) in zip(axes, vars_to_plot): #[HERE!!!]
    for region, marker in [('nearshore', 'o'), ('offshore', 's')]:
        d = season_space[season_space['shore_bin'] == region]
        ax.plot(d['season'], d[v], marker=marker, label=region)
    ax.set_title(title)
    ax.grid(alpha=0.3)

axes[0].legend()
plt.tight_layout() #keeps on clipping :/
plt.show()

ALSO: I'd like to check out how many low-O2 observaions occur in each seasonal and spatial category:

In [ ]:
pivot_lo = season_space.pivot(index='season', columns='shore_bin', values='low_oxygen_freq')
# pandas daaframe pivot changes data from a long to wide format for comp)

pivot_lo.plot(kind='bar', figsize=(7, 4))
plt.ylabel('Fraction with Bottom DO < 3 mg/L')
plt.title('Low-Oxygen Frequency by Season and Distance from Shore')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.show()

Finally, it is useful to look at the continuous relationship directly: distance from shore versus bottom oxygen, with chlorophyll alpha concentration vsualized by color ramp:

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    gdf['distance_to_shore_m'],
    gdf['bottom_do_mg_l'],
    c=gdf['chl_a'],
    s=10
)

plt.xlabel('Distance from Shore (m)')
plt.ylabel('Bottom DO (mg/L)')
plt.title('Distance from Shore vs Bottom Dissolved Oxygen')
plt.colorbar(label='Chlorophyll-a')
plt.grid(alpha=0.3)
plt.show()

#### Summary

Overall, the summer observations appear to be more strongly associated with higher chlorophyll values and more frequent low-oxygen conditions than the winter observations. Nearshore points also appear more likely to show reduced bottom dissolved oxygen than offshore points. 

This broadly supports the idea that warm-season nearshore phytoplankton-rich conditions correspond more closely with hypoxic events than cooler, farther-from-shore conditions. At the same time, chlorophyll-a is only being used here as a general phytoplankton proxy, so these patterns should not be read as direct evidence of cyanobacteria or diatom dominance by themselves...
